# Barrier Crossing Analysis

This notebook is derived from `compare_metrics_&_cluster_change.ipynb`. It retains the workflow needed for the CDE/cluster-score-filtered ML accuracy analysis and saves the output tables to the `barrier_jump` subfolder.

## Setup

Import all required libraries and configure device and precision settings.

In [ ]:
import sys
import gc
import os
import re
from pathlib import Path
import pandas as pd


sys.path.append("../../src/")

import torch
import torch.nn as nn
from torch import topk
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde
import seaborn as sns

from adabmDCA.fasta import get_tokens, write_fasta, import_from_fasta, encode_sequence
from adabmDCA.utils import get_device, get_dtype
from adabmDCA.dataset import DatasetDCA
from adabmDCA.functional import one_hot
from adabmDCA.statmech import compute_energy
from adabmDCA.io import load_params

import arDCA_paths
from arDCA_paths import arDCA_paths

device = get_device("cpu")
device2 = get_device("cuda")
dtype = get_dtype("float32")

plt.rcParams.update({
    "text.usetex": True,
})

print("Libraries loaded successfully!")

## Required Functions

Define prediction strategies (`predict_naive`, `predict`, `predict_cond_start`, `predict_cond_end`) and the accuracy metric. Each function operates on one-hot encoded sequence triplets `(S_start, S_mid, S_end)` and returns predictions for the third segment.

In [ ]:
def predict_naive(data):
    X = one_hot(data.clone(), num_classes=21).to(dtype=dtype)
    B, L, q = X.shape
    l = L // 3
    first, second = X[:, :l, :], X[:, l:2*l, :]
    third = first.clone()
    mask_equal = (first == second).all(dim=2)
    rand = torch.rand(B, l, device=X.device)
    mask_replace = (~mask_equal) & (rand >= 0.5)
    third[mask_replace] = second[mask_replace]
    X[:, 2*l:3*l, :] = third
    return X.argmax(dim=-1)

def predict(data, model, ML=False, beta=1, device="cpu"):
    with torch.no_grad():
        model.to(device)
        X = one_hot(data.clone(), num_classes=21).to(dtype=dtype).to(device)
        L = X.shape[1]
        l = model.L // 3
        for i in range(L - l, L):
            prob = model.forward(X[:, :i, :], beta=beta)
            sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()
            X[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)
        model.to("cpu")
        return X.argmax(dim=-1).to("cpu")


def predict_cond_start(data, model, ML=False, beta=1, device="cpu"):
    """Predict using arDCA by sequentially filling the last third of the sequence."""
    torch.no_grad()
    model.to(device)
    # X = data.clone().to(device)
    L = data.shape[1]
    l = L // 3
    X_end = one_hot(data, num_classes=21)[:, 2*l:, :].clone().to(dtype=dtype).to(device)
    X_use = one_hot(data, num_classes=21)[:, :2*l, :].clone().to(dtype=dtype).to(device)
    for i in range(l, 2*l):
        prob = model.forward(X_use[:, :i, :], beta=beta)
        sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()
        X_use[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)
    model.to("cpu")
    X = torch.cat((X_use[:, :l], X_end, X_use[:, l:]), dim=1)
    return X.argmax(dim=-1).to("cpu")

def predict_cond_end(data, model, ML=False, beta=1, device="cpu"):
    """Predict using arDCA by sequentially filling the last third of the sequence."""
    torch.no_grad()
    model.to(device)
    L = data.shape[1]
    l = L // 3
    X_start = one_hot(data, num_classes=21)[:, :l, :].clone().to(dtype=dtype).to(device)
    X_use = one_hot(data, num_classes=21)[:, l:, :].clone().to(dtype=dtype).to(device)
    for i in range(l, 2*l):
        prob = model.forward(X_use[:, :i, :], beta=beta)
        sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()
        X_use[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)
    model.to("cpu")
    X = torch.cat((X_start, X_use), dim=1)
    return X.argmax(dim=-1).to("cpu")

def compute_accuracy(data, prediction):
    l = data.shape[1] // 3
    third = data[:, 2*l:3*l, :].argmax(dim=-1)
    third_pred = prediction[:, 2*l:3*l, :].argmax(dim=-1)
    return (third == third_pred).float().sum(dim=-1) / l


## Path Configuration

Select the protein family and set all relevant input/output paths: data directory, CDE arrays, MSA file, bmDCA model, trained arDCA models, and output directory.

In [ ]:
protein_family = "PF00072"
# use_1cond = True
best = ""
patience = "patience5/"
percentile = 0

if protein_family == "Chorismate Mutase":
    data_path = "../generated_data/CM"
    cde_path = "../generated_data/CM/full_cde_test"
    MSA_path = "../MSAs/CM_130530_MC.fasta"
    original_model_path = "../evolution_bmDCA_model/CM/params.dat"
    t = 0
    model_paths = f"../models_train_val/CM/{patience}"
    model_1cond_path = f"../models_train_val/CM/{patience}1cond/"
   
    reg = "rJ1e-3_rH1e-5" # "long_run_rJ1e-3_rH1e-5"
    
    if percentile == 0:
        output_path = f"../immagini_paper/CM/{patience}compare_metrics/{best}_{reg}/"
    else: 
        output_path = f"../immagini_paper/CM/{patience}compare_metrics/{best}_{reg}_percentile{percentile}/"


elif protein_family == "PF00072":
    data_path = "../generated_data/PF00072"
    cde_path = "../generated_data/PF00072/full_cde_test"
    MSA_path = "../MSAs/PF00072.fasta"
    original_model_path = "../evolution_bmDCA_model/PF00072/params.dat"
    t = 0
    model_paths = f"../models_train_val/PF00072/{patience}"
    model_1cond_path = f"../models_train_val/PF00072/{patience}1cond/"
    reg = "rJ1e-3_rH1e-5" # "long_run_rJ1e-3_rH1e-5"
    if percentile == 0:
        output_path = f"../immagini_paper/PF00072/{patience}compare_metrics/{best}_{reg}/"
    else:
        output_path = f"../immagini_paper/PF00072/{patience}compare_metrics/{best}_{reg}_percentile{percentile}/"


elif protein_family == "betalactamase":
    data_path = "../generated_data/betalactamase"
    cde_path = "../generated_data/betalactamase/full_cde_test"
    MSA_path = "../MSAs/betalactamase_nodupl_natural_noflankgaps.fa"
    original_model_path = "../evolution_bmDCA_model/betalactamase/params.dat"
    t = 0
    model_paths = f"../models_train_val/betalactamase/{patience}"
    model_1cond_path = f"../models_train_val/betalactamase/{patience}1cond/"
    reg = "rJ1e-3_rH1e-5" # "long_run_rJ1e-3_rH1e-5"

    if percentile == 0:
        output_path = f"../immagini_paper/betalactamase/{patience}compare_metrics/{best}_{reg}/"
    else:
        output_path = f"../immagini_paper/betalactamase/{patience}compare_metrics/{best}_{reg}_percentile{percentile}/"

else:
    raise ValueError(f"Unknown protein family: {protein_family}")


# Specific addition for this analysis: output goes to barrier_jump subfolder.
output_path = str(Path(output_path) / "barrier_jump") + "/"

os.makedirs(output_path, exist_ok=True)
print(f"Output path created/verified: {output_path}")


## Data Loading

Load the test FASTA datasets for all timescales, convert to integer-encoded tensors, and build one-hot representations. Up to `n_seqs` sequences are loaded per timescale.

In [ ]:
data_type = "ACDEFGHIKLMNPQRSTVWY-"
tokens = get_tokens(data_type)

timescales = ["10e1", "10e2", "10e3", "10e4", "10e5", "10e6"] 
n_seqs = 20_000

data_files = [
    f"{data_path}/10e1_test.fasta",
    f"{data_path}/10e2_test.fasta",
    f"{data_path}/10e3_test.fasta",
    f"{data_path}/10e4_test.fasta",
    f"{data_path}/10e5_test.fasta",
    f"{data_path}/10e6_test.fasta",
]

datasets = {}
dataset_indices = {}
for t, path in zip(timescales, data_files):
    ds = DatasetDCA(path_data=path, alphabet=data_type, device=device, dtype=dtype, no_reweighting=True)
    # indices = torch.randperm(ds.data.shape[0])[:n_seqs]
    # dataset_indices[t] = indices.cpu()
    datasets[t] = ds.data[:n_seqs] #.argmax(dim=-1)
    del ds
    gc.collect()

print(f"Loaded {len(datasets)} datasets with {n_seqs} sequences each")

datasets_oh = {
    key: one_hot(data, num_classes=21).to(dtype=dtype)
    for key, data in datasets.items()
}

## Natural MSA and Original bmDCA Model

Load the natural Multiple Sequence Alignment (MSA) and the reference bmDCA model parameters. Derive the per-segment length `L` from the dataset shape.

In [ ]:
q = 21

headers, msa_enc_nat = import_from_fasta(MSA_path, tokens=tokens, filter_sequences=True)
msa_oh_nat = one_hot(torch.tensor(msa_enc_nat, device=device, dtype=torch.int32), num_classes=q).to(dtype)

M, L3 = datasets[timescales[0]].shape
L = L3 // 3

original_tokens = get_tokens("ACDEFGHIKLMNPQRSTVWY-")
original_model = load_params(original_model_path, tokens=original_tokens, device=device, dtype=dtype)

print(f"Loaded natural MSA: {msa_oh_nat.shape[0]} sequences, L={L}")

## CDE Threshold

Define the `load_cde_dict` utility, then load the precomputed CDE arrays for the initial segment (`CDE_0`, `CDE_0_mean`). Compute a global median threshold over all timescales to classify sequences by evolutionary constraint.

In [ ]:
def load_cde_dict(path, datasets=datasets, timescales=timescales, dtype=dtype, dataset_indices=globals().get("dataset_indices", None)):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"CDE file not found: {path}")
    loaded = np.load(path, allow_pickle=True).item()
    missing = [key for key in timescales if key not in loaded]
    if missing:
        raise KeyError(f"Missing timescales in {path}: {missing}")

    cde = {}
    for key in timescales:
        values = loaded[key]
        if dataset_indices is not None and key in dataset_indices:
            indices = np.asarray(dataset_indices[key].cpu().numpy(), dtype=np.int64)
            if values.shape[0] <= indices.max():
                raise ValueError(
                    f"CDE file {path} for {key} has {values.shape[0]} values, "
                    f"but dataset uses index {indices.max()}."
                )
            selected = values[indices]
        else:
            n_values = datasets[key].shape[0]
            if values.shape[0] < n_values:
                raise ValueError(
                    f"CDE file {path} for {key} has only {values.shape[0]} values, "
                    f"but dataset has {n_values} sequences."
                )
            selected = values[:n_values]
        cde[key] = torch.as_tensor(selected, dtype=dtype)
    return cde

# (was: if use_1cond:)
cde_path = Path(cde_path)
CDE_start = load_cde_dict(cde_path / "CDE_0.npy")
CDE_start_mean = load_cde_dict(cde_path / "CDE_0_mean.npy")

all_cde_start_values = torch.cat([CDE_start_mean[t] for t in timescales])
soglia_cde_iniziale = torch.quantile(all_cde_start_values, 0.5).item()

print(f"\n{'='*70}")
print("DEFINIZIONE SOGLIA CDE")
print(f"{'='*70}")
print(f"50° percentile di CDE_start_mean: {soglia_cde_iniziale:.4f}")
print(f"{'='*70}\n")


## arDCA Models and Predictions

Load one trained arDCA model per timescale, then generate reconstructions of the third segment using three strategies: naive (direct-path heuristic), arDCA stochastic sampling, and arDCA ML (greedy) decoding. Convert all predictions to one-hot format.

In [ ]:
models = {}
for t in timescales:
    path_params = model_paths
    params = torch.load(model_paths + f"{t}_{reg}/params{best}.pth", map_location='cpu')
    params['entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    params['inverse_entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    model = arDCA_paths(L=L3, q=q).to(device='cpu', dtype=dtype)
    model.load_state_dict(params, strict=False)
    models[t] = model

print(f"Loaded {len(models)} arDCA models")

In [ ]:
Temperature = 1.0
beta = 1 / Temperature

print("Predicting with naive...")
predicted_naive = {key: predict_naive(datasets[key]) for key in timescales}

print("Predicting with arDCA (sampling)...")
predicted = {key: predict(data, models[key], ML=False, beta=beta, device=device2) for key, data in datasets.items()}

print("Predicting with arDCA (ML)...")
predictedML = {key: predict(data, models[key], ML=True, beta=beta, device=device2) for key, data in datasets.items()}


predicted_naive_oh = {
    key: one_hot(data, num_classes=21).to(dtype=dtype)
    for key, data in predicted_naive.items()
}
predicted_oh = {
    key: one_hot(data, num_classes=21).to(dtype=dtype)
    for key, data in predicted.items()
}
predictedML_oh = {
    key: one_hot(data, num_classes=21).to(dtype=dtype)
    for key, data in predictedML.items()
}


## ML Accuracy

Compute per-sequence reconstruction accuracy for the three prediction methods across all timescales.

In [ ]:
acc_naive = {key: compute_accuracy(datasets_oh[key], predicted_naive_oh[key]) for key in timescales}
acc_mdl = {key: compute_accuracy(datasets_oh[key], predicted_oh[key]) for key in timescales}
acc_mdlML = {key: compute_accuracy(datasets_oh[key], predictedML_oh[key]) for key in timescales}
print("Accuracy computations complete!")


## CDE Barrier Crossing Analysis

Compare two sequence classes defined by global percentiles of the CDE distributions across the three segments: initial (`CDE_0`), intermediate (`CDE_T2`), and final (`CDE_T`).

- **Constrained Path**: all three segments have low CDE (below the 40th percentile).
- **Barrier Crossing**: initial and final segments have low CDE, but the intermediate segment has high CDE (above the 60th percentile), indicating a transient excursion through an energetically unfavourable region.

### 1. Load CDE for the Third Segment

Load the precomputed CDE arrays for the intermediate (`CDE_T2`) and final (`CDE_T`) segments, reusing the same `load_cde_dict` function to maintain alignment with the datasets.

In [ ]:
# Load CDE values for the third segment and, if needed, the CDE_T values used in the new analysis.
# Use the same load_cde_dict function defined above to keep alignment with datasets and timescales identical.
CDE_T2 = load_cde_dict(cde_path / "CDE_T2.npy")
CDE_T2_mean = load_cde_dict(cde_path / "CDE_T2_mean.npy")

if "CDE" not in globals():
    CDE = load_cde_dict(cde_path / "CDE_T.npy")
if "CDE_mean" not in globals():
    CDE_mean = load_cde_dict(cde_path / "CDE_T_mean.npy")

print("CDE disponibili per la nuova analisi:")
print(f"  CDE_0_mean timescales:  {list(CDE_start_mean.keys())}")
print(f"  CDE_T2_mean timescales: {list(CDE_T2_mean.keys())}")
print(f"  CDE_T_mean timescales:  {list(CDE_mean.keys())}")


### 2. Global Percentiles and Accuracy Table

Compute global p40 and p60 thresholds from the pooled CDE distributions of all three segments across all timescales. Assign each sequence to one of the two classes and compute mean ML accuracy per class and timescale. Save the summary table as CSV.

In [ ]:
# Concatenate CDE_0, CDE_T2 and CDE_T across all timescales (in this order)
# and compute a single pair of global thresholds p40/p60.
all_cde_values = torch.cat(
    [torch.cat([CDE_start_mean[t] for t in timescales]),
     torch.cat([CDE_T2_mean[t] for t in timescales]),
     torch.cat([CDE_mean[t] for t in timescales])]
)

p40 = torch.quantile(all_cde_values, 0.40).item()
p60 = torch.quantile(all_cde_values, 0.60).item()

print(f"p40 globale: {p40:.4f}")
print(f"p60 globale: {p60:.4f}")

barrier_class_rows = []
barrier_samples_by_timescale = {}

for t in timescales:
    accML = acc_mdlML[t].cpu().numpy()
    cde_0 = CDE_start_mean[t].cpu().numpy()
    cde_t2 = CDE_T2_mean[t].cpu().numpy()
    cde_t = CDE_mean[t].cpu().numpy()

    # Class 1: all three segments have low CDE.
    class_1_mask = (cde_0 < p40) & (cde_t2 < p40) & (cde_t < p40)

    # Class 2: low CDE on first and T segments, but high CDE on the third (intermediate).
    class_2_mask = (cde_0 < p40) & (cde_t2 > p60) & (cde_t < p40)

    barrier_samples_by_timescale[t] = {
        "CDE0_low_CDET2_low_CDET_low": class_1_mask,
        "CDE0_low_CDET2_high_CDET_low": class_2_mask,
    }

    barrier_class_rows.append({
        "Timescale": t,
        "Classe": "CDE_0 < p40, CDE_T2 < p40, CDE_T < p40",
        "Accuracy ML media": accML[class_1_mask].mean() if class_1_mask.sum() > 0 else np.nan,
        "N samples": int(class_1_mask.sum()),
    })
    barrier_class_rows.append({
        "Timescale": t,
        "Classe": "CDE_0 < p40, CDE_T2 > p60, CDE_T < p40",
        "Accuracy ML media": accML[class_2_mask].mean() if class_2_mask.sum() > 0 else np.nan,
        "N samples": int(class_2_mask.sum()),
    })

barrier_accuracy_table = pd.DataFrame(barrier_class_rows)
barrier_accuracy_table_path = output_path + "barrier_cde_accuracy_by_timescale.csv"
barrier_accuracy_table.to_csv(barrier_accuracy_table_path, index=False)

print(barrier_accuracy_table.to_string(index=False))
print(f"\nTabella salvata: {barrier_accuracy_table_path}")


### 3. Combined Dataset by Hamming Bins

Pool sequences from all timescales, keeping the two CDE classes separate. Compute the Hamming distance between the first and second segments and bin sequences into 10 equal-width intervals. Discard bins with fewer than `min_points_per_violin` samples.

In [ ]:
# Merge all timescales while keeping the two classes separate.
# Hamming distance is computed between first and second segments as the number of differing positions.
barrier_plot_rows = []
class_labels = {
    "CDE0_low_CDET2_low_CDET_low": r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$",
    "CDE0_low_CDET2_high_CDET_low": r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$",
}

for t in timescales:
    accML = acc_mdlML[t].cpu().numpy()
    sequences = datasets_oh[t].cpu().numpy()

    first_seg = sequences[:, :L, :].argmax(axis=2)
    second_seg = sequences[:, L:2*L, :].argmax(axis=2)
    hamming_0_t = (first_seg != second_seg).sum(axis=1)

    for class_key, mask in barrier_samples_by_timescale[t].items():
        selected_indices = np.where(mask)[0]
        for idx in selected_indices:
            barrier_plot_rows.append({
                "Timescale": t,
                "Classe": class_labels[class_key],
                "Hamming first-second": int(hamming_0_t[idx]),
                "Accuracy ML": float(accML[idx]),
            })

barrier_plot_df = pd.DataFrame(barrier_plot_rows)
min_points_per_violin = 10

if barrier_plot_df.empty:
    barrier_plot_df_filtered = pd.DataFrame()
    hamming_bin_order = []
    print("Nessun sample nelle due classi selezionate: plot non generato.")
else:
    min_hamming = int(barrier_plot_df["Hamming first-second"].min())
    max_hamming = int(barrier_plot_df["Hamming first-second"].max())

    # Divide the full observed Hamming distance range into 10 bins.
    # The +1 on the maximum ensures sequences with exactly max_hamming are included.
    if min_hamming == max_hamming:
        bin_edges = np.array([min_hamming, max_hamming + 1])
    else:
        bin_edges = np.linspace(min_hamming, max_hamming + 1, 10)

    barrier_plot_df["Hamming bin"] = pd.cut(
        barrier_plot_df["Hamming first-second"],
        bins=bin_edges,
        right=False,
        include_lowest=True,
        duplicates="drop",
    )

    # Preserve the natural bin order even after filtering by minimum count.
    hamming_bin_intervals = list(barrier_plot_df["Hamming bin"].cat.categories)
    hamming_bin_order = [str(interval) for interval in hamming_bin_intervals]
    def format_hamming_interval(interval):
        left = int(round(interval.left))
        right = int(round(interval.right))
        return f"{left}-{right}"

    hamming_bin_tick_labels_by_bin = {
        str(interval): format_hamming_interval(interval)
        for interval in hamming_bin_intervals
    }
    barrier_plot_df["Hamming bin"] = barrier_plot_df["Hamming bin"].astype(str)

    group_counts = (
        barrier_plot_df
        .groupby(["Hamming bin", "Classe"], observed=True)
        .size()
        .reset_index(name="N samples")
    )
    valid_groups = group_counts[group_counts["N samples"] >= min_points_per_violin]

    barrier_plot_df_filtered = barrier_plot_df.merge(
        valid_groups[["Hamming bin", "Classe"]],
        on=["Hamming bin", "Classe"],
        how="inner",
    )
    hamming_bin_order = [
        bin_label for bin_label in hamming_bin_order
        if bin_label in set(barrier_plot_df_filtered["Hamming bin"])
    ]
    hamming_bin_tick_labels = [
        hamming_bin_tick_labels_by_bin[bin_label]
        for bin_label in hamming_bin_order
    ]

    barrier_plot_table_path = output_path + "barrier_cde_accuracy_hamming_bins_data.csv"
    # barrier_plot_df.to_csv(barrier_plot_table_path, index=False)

    barrier_plot_filtered_path = output_path + "barrier_cde_accuracy_hamming_bins_data_min10.csv"
    # barrier_plot_df_filtered.to_csv(barrier_plot_filtered_path, index=False)

    removed_groups = len(group_counts) - len(valid_groups)
    print(f"Dati completi per il plot salvati: {barrier_plot_table_path}")
    print(f"Dati filtrati per il plot salvati: {barrier_plot_filtered_path}")
    print(f"Samples totali prima del filtro: {len(barrier_plot_df)}")
    print(f"Samples totali nel plot dopo filtro min {min_points_per_violin}: {len(barrier_plot_df_filtered)}")
    print(f"Gruppi bin/classe esclusi per N < {min_points_per_violin}: {removed_groups}")


### 4. ML Accuracy Distribution by Hamming Bin (Aggregated)

Plot horizontal histogram distributions of ML reconstruction accuracy for each Hamming bin and CDE class. A horizontal black bar marks the within-group mean. Save as PDF.

In [ ]:
if barrier_plot_df_filtered.empty:
    print(f"Nessun gruppo bin/classe con almeno {min_points_per_violin} dati: plot non generato.")
else:
    from matplotlib.patches import Patch

    sns.set_theme(style="whitegrid", context="talk")

    n_bins_visible = max(1, barrier_plot_df_filtered["Hamming bin"].nunique())
    fig_width = max(12, min(22, 1.25 * n_bins_visible + 5))
    fig, ax = plt.subplots(figsize=(fig_width, 7.5))

    n_bins = 30
    hist_alpha = 0.7
    bar_edge_linewidth = 0.5
    mean_linewidth = 3
    histogram_width = 0.36
    histogram_offsets = {
        r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$": -0.22,
        r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$": 0.22,
    }
    accuracy_bins = np.linspace(0.0, 1.0, n_bins + 1)

    palette = {
        r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$": "#4C78A8",
        r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$": "#F58518",
    }
    hue_order = [r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$", 
                r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$"]

    for i, hamming_bin in enumerate(hamming_bin_order):
        for class_label in hue_order:
            data_vals = barrier_plot_df_filtered.loc[
                (barrier_plot_df_filtered["Hamming bin"] == hamming_bin)
                & (barrier_plot_df_filtered["Classe"] == class_label),
                "Accuracy ML",
            ].dropna().to_numpy()

            if len(data_vals) == 0:
                continue

            counts, bin_edges = np.histogram(data_vals, bins=accuracy_bins, density=True)
            max_count = counts.max()
            counts_normalized = counts / max_count * histogram_width * 0.8 if max_count > 0 else np.zeros_like(counts)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            offset = histogram_offsets[class_label]

            for bc, cn in zip(bin_centers, counts_normalized):
                if cn > 0:
                    ax.barh(
                        bc,
                        cn,
                        height=(bin_edges[1] - bin_edges[0]) * 0.9,
                        left=i + offset - cn / 2,
                        color=palette[class_label],
                        alpha=hist_alpha,
                        edgecolor="black",
                        linewidth=bar_edge_linewidth,
                    )

            mean_val = data_vals.mean()
            ax.hlines(
                mean_val,
                i + offset - histogram_width * 0.4,
                i + offset + histogram_width * 0.4,
                colors="black",
                linewidth=mean_linewidth,
                zorder=10,
            )

    # ax.set_title(
    #     f"Accuracy ML per Hamming distance e classe CDE ($N \geq {min_points_per_violin}$)",
    #     pad=16,
    #     fontsize=18,
    #     weight="bold",
    # )
    ax.set_xticks(np.arange(len(hamming_bin_order)))
    ax.set_xticklabels(hamming_bin_tick_labels, fontsize=20)
    ax.set_xlabel(r"$d(S_\mathrm{start}, S_\mathrm{end})$", labelpad=12, fontsize=24)
    ax.set_ylabel("Reconstruction Accuracy (ML greedy decoding)", labelpad=10, fontsize=24)
    ax.set_ylim(-0.02, 1.02)
    ax.tick_params(axis="x", labelsize=20) #, fontsize=20)
    ax.tick_params(axis="y", labelsize=20)
    ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.45)
    ax.grid(axis="x", visible=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    handles = [
        Patch(facecolor=palette[label], edgecolor="black", alpha=hist_alpha, label=label)
        for label in hue_order
    ]
    labels = hue_order
    ax.legend(
        handles,
        labels,
        # title="Classe",
        frameon=False,
        fancybox=False,
        edgecolor="0.85",
        loc="upper right",
        fontsize=20,
    )



    fig.tight_layout()

    barrier_plot_path = output_path + "barrier_cde_accuracy_by_hamming_bins.pdf"
    fig.savefig(barrier_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Plot salvato: {barrier_plot_path}")


### 5. ML Accuracy Distribution by Hamming Bin (Per Timescale)

Repeat the aggregated plot above split by timescale. Each distribution is shown only when the `(timescale, bin, class)` group contains at least `min_points_per_violin` sequences.

In [ ]:
# Same plot as the aggregated panel, but split by timescale.
# Each distribution is shown only if the group (timescale, Hamming bin, class) has at least 10 samples.
if barrier_plot_df.empty:
    print("Nessun dato disponibile: plot per timescale non generato.")
else:
    group_counts_by_timescale = (
        barrier_plot_df
        .groupby(["Timescale", "Hamming bin", "Classe"], observed=True)
        .size()
        .reset_index(name="N samples")
    )
    valid_groups_by_timescale = group_counts_by_timescale[
        group_counts_by_timescale["N samples"] >= min_points_per_violin
    ]

    barrier_plot_df_by_timescale = barrier_plot_df.merge(
        valid_groups_by_timescale[["Timescale", "Hamming bin", "Classe"]],
        on=["Timescale", "Hamming bin", "Classe"],
        how="inner",
    )

    barrier_plot_by_timescale_path = output_path + "barrier_cde_accuracy_hamming_bins_by_timescale_data_min10.csv"
    # barrier_plot_df_by_timescale.to_csv(barrier_plot_by_timescale_path, index=False)

    if barrier_plot_df_by_timescale.empty:
        print(f"Nessun gruppo timescale/bin/classe con almeno {min_points_per_violin} dati: plot non generato.")
    else:
        from matplotlib.patches import Patch

        sns.set_theme(style="whitegrid", context="talk")

        n_bins = 30
        hist_alpha = 0.7
        bar_edge_linewidth = 0.5
        mean_linewidth = 3
        histogram_width = 0.36
        histogram_offsets = {
            r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$": -0.22,
            r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$": 0.22,
        }
        accuracy_bins = np.linspace(0.0, 1.0, n_bins + 1)

        palette = {
            r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$": "#4C78A8",
            r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$": "#F58518",
        }
        hue_order = [r"Constrained Path: $\overline{\mathrm{CDE}}(S_\mathrm{mid})<p40$", r"Barrier Crossing: $\overline{\mathrm{CDE}}(S_\mathrm{mid})>p60$"]

        n_cols = 3
        n_rows = int(np.ceil(len(timescales) / n_cols))
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(22, 5.8 * n_rows),
            sharey=True,
            sharex=True,
            constrained_layout=True,
        )
        axes = np.asarray(axes).reshape(-1)

        for ax, t in zip(axes, timescales):
            df_t = barrier_plot_df_by_timescale[barrier_plot_df_by_timescale["Timescale"] == t].copy()

            if df_t.empty:
                ax.text(
                    0.5,
                    0.5,
                    f"{t}\nNessun gruppo con $N \geq {min_points_per_violin}$",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=13,
                    color="0.35",
                )
                ax.set_axis_off()
                continue

            for i, hamming_bin in enumerate(hamming_bin_order):
                for class_label in hue_order:
                    data_vals = df_t.loc[
                        (df_t["Hamming bin"] == hamming_bin) & (df_t["Classe"] == class_label),
                        "Accuracy ML",
                    ].dropna().to_numpy()

                    if len(data_vals) == 0:
                        continue

                    counts, bin_edges = np.histogram(data_vals, bins=accuracy_bins, density=True)
                    max_count = counts.max()
                    counts_normalized = counts / max_count * histogram_width * 0.8 if max_count > 0 else np.zeros_like(counts)
                    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                    offset = histogram_offsets[class_label]

                    for bc, cn in zip(bin_centers, counts_normalized):
                        if cn > 0:
                            ax.barh(
                                bc,
                                cn,
                                height=(bin_edges[1] - bin_edges[0]) * 0.9,
                                left=i + offset - cn / 2,
                                color=palette[class_label],
                                alpha=hist_alpha,
                                edgecolor="black",
                                linewidth=bar_edge_linewidth,
                            )

                    mean_val = data_vals.mean()
                    ax.hlines(
                        mean_val,
                        i + offset - histogram_width * 0.4,
                        i + offset + histogram_width * 0.4,
                        colors="black",
                        linewidth=mean_linewidth,
                        zorder=10,
                    )

            ax.set_xticks(np.arange(len(hamming_bin_order)))
            ax.set_xticklabels(hamming_bin_tick_labels, rotation=40, fontsize=22)
            ax.set_title(f"{t}", fontsize=24, weight="bold", pad=10)
            ax.set_xlabel("")
            ax.set_ylabel("Recnstruction Accuracy (ML greedy decoding)" if ax in axes[::n_cols] else "", fontsize=22)
            ax.set_ylim(-0.02, 1.02)
  
            ax.tick_params(axis="x", rotation=40, labelsize=22) 
        
            ax.tick_params(axis="y", labelsize=22)
            ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.4)
            ax.grid(axis="x", visible=False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)


        for ax in axes[len(timescales):]:
            ax.set_axis_off()

        handles = [
            Patch(facecolor=palette[label], edgecolor="black", alpha=hist_alpha, label=label)
            for label in hue_order
        ]
        labels = hue_order

        fig.legend(
            handles,
            labels,
            # title="Classe",
            loc="upper center",
            ncol=2,
            frameon=True,
            fancybox=False,
            edgecolor="0.85",
            bbox_to_anchor=(0.5, 1.1),
            fontsize=22
        )
     
        fig.supxlabel(r"$d(S_\mathrm{start}, S_\mathrm{mid})$", fontsize=24)

        barrier_timescale_plot_path = output_path + "barrier_cde_accuracy_by_hamming_bins_by_timescale.pdf"
        fig.savefig(barrier_timescale_plot_path, dpi=300, bbox_inches="tight")
        plt.show()

        removed_groups_by_timescale = len(group_counts_by_timescale) - len(valid_groups_by_timescale)
        print(f"Dati filtrati per plot timescale salvati: {barrier_plot_by_timescale_path}")
        print(f"Gruppi timescale/bin/classe esclusi per N < {min_points_per_violin}: {removed_groups_by_timescale}")
        print(f"Plot timescale salvato: {barrier_timescale_plot_path}")
